## Data and cash flows

**Conventions**:
- `coupon` is in **percent** (5 = 5%), matching the bond table's `Coupon (%)` column.
- Yields and spreads are **decimals** (0.0488 = 4.88%)
- Time is in **years**
- Face defaults to 100.


In [ ]:
import sys
from datetime import date, datetime
from functools import lru_cache
from numbers import Real
from pathlib import Path

import numpy as np
import pandas as pd

# fred_api.py lives in the repo root, one folder up from this notebook
sys.path.insert(0, str(Path.cwd().parent))
from fred_api import get_latest


### load_bonds(csv)
Reads the finance students' bond table.
format: Issuer | CUSIP | Coupon (%) | Maturity Date | Price | Rating
csv is the path to the csv file. Maturity is parsed to a date so the other functions can use it.


In [ ]:
def load_bonds(csv):
    bonds = pd.read_csv(csv)
    bonds.columns = bonds.columns.str.strip()
    # normilize the date format
    bonds["Maturity Date"] = pd.to_datetime(bonds["Maturity Date"]).dt.date
    return bonds


### _to_date(d)
Helper. Turns a `"YYYY-MM-DD"` string, `date`, `datetime`, pandas `Timestamp` or numpy `datetime64` into a plain `date`.
`datetime` is checked before `date` because `datetime` is a subclass of `date`.

In [ ]:
def _to_date(d):
    if isinstance(d, str):
        return datetime.strptime(d, "%Y-%m-%d").date()
    if isinstance(d, datetime):  # check datetime first: datetime is a subclass of date
        return d.date()
    if isinstance(d, date):
        return d
    return pd.Timestamp(d).date()  # pandas Timestamp, numpy datetime64

### _shift_years(d, n)
Helper. Moves date `d` by `n` calendar years (negative `n` goes back).
Feb 29 becomes Feb 28 in a non-leap year.

In [ ]:
def _shift_years(d, n):
    """d moved n calendar years. Feb 29 falls back to Feb 28 in non-leap years."""
    try:
        return d.replace(year=d.year + n)
    except ValueError:
        return d.replace(year=d.year + n, day=28)

### years_to_maturity(maturity_date, today)
Converts a date into years with an exact calendar count, not days / 365.25.
Whole years are counted back from maturity (so 2026-11-10 to 2031-11-10 is exactly 5.0). The leftover days are
divided by the actual length of the year they fall in (365 or 366). This is the same anchor-on-maturity logic as
`cash_flows`, close to the actual/actual convention used for Treasuries.
Accepts dates, datetimes or `"YYYY-MM-DD"` strings. `today` defaults to the current date.


In [ ]:
def years_to_maturity(maturity_date, today=None):
    """Exact calendar years left: whole years counted back from maturity, plus the
    leftover days divided by the true length (365 or 366) of the year they fall in."""
    today = date.today() if today is None else _to_date(today)
    maturity = _to_date(maturity_date)
    if maturity < today:
        raise ValueError(f"Bond matured {(today - maturity).days} days before {today}.")

    whole = maturity.year - today.year
    while _shift_years(maturity, -whole) < today:  # step back until the anniversary is on/after today
        whole -= 1
    anniversary = _shift_years(maturity, -whole)  # first anniversary of maturity on/after today
    year_before = _shift_years(maturity, -(whole + 1))
    stub = (anniversary - today).days / (anniversary - year_before).days
    return whole + stub

### cash_flows(coupon, maturity, face=100)
Lists each semiannual coupon payment plus the final repayment.
`maturity` can be years (float) or a maturity date. Payments are counted **back from maturity** in 0.5-year
steps, so a bond with 4.8 years left has its first coupon in 0.3 years, not 0.5.
- `maturity` given as a number, including numpy numbers like `np.int64`, is read as years.
- Maturing today (0 years left) gives one row at `years=0`: the final coupon plus principal.

In [ ]:
def cash_flows(coupon, maturity, face: float | None = 100.0, today=None, freq=2):
    if face == None:
        face = 0

    # Real (not int/float) so numpy numbers like np.int64 from a DataFrame count as years, not dates
    years = maturity if isinstance(maturity, Real) else years_to_maturity(maturity, today)
    step = 1 / freq

    n = int(np.ceil(round(years * freq, 9)))  # round() stops 10.0 * 2 = 19.999... adding a period
    n = max(n, 1)  # maturing today (years=0): one final coupon + principal paid now

    # Needs times to go back words so you can figure out when the next cupon payment is
    # Ex Semi-anual cupons 5 year maturity .2 years in
    # Backwords: 4.8 -> 4.2 -> ... -> 0.2=next cupon payment
    times = years - step * np.arange(n)[::-1]  # oldest first, last one = maturity
    flows = np.full(n, face * coupon / 100 / freq)
    flows[-1] += face  # principal repaid with the last coupon
    
    return pd.DataFrame({"years": times, "cash_flow": flows})


### present_value(cf, ytm, compounding=2)
Discounts the rows from `cash_flows` back to today and adds them up, giving the bond's (dirty) price.
`ytm` is a decimal (0.05 = 5%). `compounding` is how often the yield compounds:
- a number `m` (1 = annual, 2 = semiannual, 4 = quarterly, 12 = monthly): $PV = \sum CF_t \,/\, (1 + y/m)^{m t}$
- `"continuous"`: $PV = \sum CF_t \, e^{-y t}$

The same yield gives a lower price with continuous compounding, because it compounds more often. To get the
same price, convert the rate: $r_{cont} = m \ln(1 + y/m)$.
Compounding is separate from coupon frequency: `cash_flows(freq=...)` sets when coupons are paid, `compounding` sets how the yield is applied.

In [ ]:
def present_value(cf, ytm, compounding=2):
    t = cf["years"].to_numpy(dtype=float)
    flows = cf["cash_flow"].to_numpy(dtype=float)

    if compounding == "continuous":
        factors = np.exp(-ytm * t)  # limit of (1 + y/m)^(-m t) as m -> infinity
    elif isinstance(compounding, Real) and compounding > 0:
        factors = (1 + ytm / compounding) ** (-compounding * t)
    else:
        raise ValueError(f'compounding must be a positive number or "continuous", got {compounding!r}')

    return float((flows * factors).sum())

### treasury_curve()
Gets the last 10 FRED prints of DGS5, DGS7 and DGS10 (`TENORS`) and keeps the latest date where all three exist.
Returns those yields as decimals, indexed by tenor in years (5, 7, 10). The date is in `.attrs["as_of"]`.
- Raises `RuntimeError` if none of the 10 dates has all three.
- Not cached: every call fetches fresh from FRED (3 requests).

In [ ]:
TENORS = {5: "DGS5", 7: "DGS7", 10: "DGS10"}


def treasury_curve():
    """Latest common-date 5/7/10Y yields as decimals, indexed by tenor in years."""
    data = pd.concat({t: get_latest(sid, n=10) for t, sid in TENORS.items()}, axis=1).dropna()
    if data.empty:
        raise RuntimeError("No date in the last 10 FRED prints has all of DGS5, DGS7, DGS10.")
    latest = data.iloc[-1] / 100  # FRED quotes percent
    latest.attrs["as_of"] = data.index[-1].date()
    return latest

### treasury_yield(years)
Pulls DGS5, DGS7 and DGS10 from FRED and linearly interpolates the Treasury yield at that maturity.
- All three come from the **same date** (latest date where all three exist).
- Outside 5-10 years it holds the end value flat and warns. Add DGS2/DGS3/DGS20 if we need a wider range.
- Calls `treasury_curve()` every time, so each call makes 3 FRED requests.


In [ ]:
def treasury_yield(years):
    curve = treasury_curve()
    lo, hi = curve.index.min(), curve.index.max()
    if not lo <= years <= hi:
        print(f"Warning: {years:.2f}y is outside {lo}-{hi}y; using flat extrapolation.")
    return float(np.interp(years, curve.index, curve.values))

### spread(bond_yield, years)
The bond's yield minus the matching Treasury yield (decimal; multiply by 10,000 for bp).
`bond_yield` will come from Brian's `ytm(price, coupon, years)`.


In [ ]:
def spread(bond_yield, years):
    return bond_yield - treasury_yield(years)


### Tests
Checks against hand math. The FRED cell needs `FRED_API_KEY` in the repo root `.env`.


In [ ]:
# years_to_maturity
assert years_to_maturity("2031-11-10", "2026-11-10") == 5.0  # exact, leap year 2028 inside
assert years_to_maturity("2027-05-10", "2026-11-10") == 181 / 365  # 181 days of a 365-day year
assert years_to_maturity("2028-11-10", "2028-05-10") == 184 / 366  # stub year contains Feb 29, 2028
assert years_to_maturity("2032-02-29", "2027-02-28") == 5.0  # leap-day maturity anchors to Feb 28
assert years_to_maturity(date(2026, 11, 10), "2026-11-10") == 0

# cash_flows: 10Y 5% bond -> 20 payments of 2.50, last one 102.50
cf = cash_flows(5, 10)
assert len(cf) == 20 and cf["cash_flow"].iloc[0] == 2.5 and cf["cash_flow"].iloc[-1] == 102.5
assert cf["cash_flow"].sum() == 150  # 20 coupons of 2.50 + 100 principal

# priced at a 5% yield (semiannual), a 5% bond is worth par
pv = (cf["cash_flow"] / (1 + 0.05 / 2) ** (2 * cf["years"])).sum()
assert abs(pv - 100) < 1e-9, pv

# off-cycle bond: 4.8 years left -> first coupon in 0.3 years, 10 payments
cf_odd = cash_flows(5, 4.8)
assert len(cf_odd) == 10 and abs(cf_odd["years"].iloc[0] - 0.3) < 1e-9

# maturity as a date
assert len(cash_flows(4.75, "2031-10-31", today="2026-11-10")) == 10
print("offline tests passed")
cash_flows(5, 2)


In [ ]:
# Live FRED: today's 5/7/10Y curve, then a sample spread
curve = treasury_curve()
print("As of", curve.attrs["as_of"], {f"{t}Y": f"{y:.2%}" for t, y in curve.items()})
assert treasury_yield(5) == curve[5] and treasury_yield(10) == curve[10]
assert abs(treasury_yield(6) - (curve[5] + curve[7]) / 2) < 1e-12  # halfway between 5Y and 7Y

s = spread(0.055, 6)  # e.g. a 6-year bond yielding 5.50%
print(f"6Y Treasury {treasury_yield(6):.3%}, spread of a 5.50% bond = {s * 1e4:.0f}bp")
